In [5]:
import sqlite3
import pandas as pd
import os

# File paths
history_db = "History"
ram_file = "datas/ram_log.csv"
output_file = "datas/browsing_history.csv"

# 1. Read RAM log timestamps
ram = pd.read_csv(ram_file)
ram["timestamp"] = pd.to_datetime(ram["timestamp"], errors="coerce")
ram = ram.dropna(subset=["timestamp"])

if ram.empty:
    raise ValueError("No valid timestamps found in the RAM log.")

ram_start = ram["timestamp"].min()
ram_end = ram["timestamp"].max()

print("RAM logging period:", ram_start, "to", ram_end)

# 2. Extract Chrome browsing history
conn = sqlite3.connect(history_db)

query = """
SELECT
    datetime((visits.visit_time / 1000000) - 11644473600,
             'unixepoch', 'localtime') AS timestamp,
    urls.url,
    urls.title
FROM visits
JOIN urls ON visits.url = urls.id
ORDER BY visits.visit_time;
"""

history = pd.read_sql_query(query, conn)
conn.close()

# 3. Convert timestamps and remove invalid rows
history["timestamp"] = pd.to_datetime(history["timestamp"], errors="coerce")
history = history.dropna(subset=["timestamp"])

# 4. Keep visits within the RAM log's overall date-time range
filtered = history[
    (history["timestamp"] >= ram_start) &
    (history["timestamp"] <= ram_end)
].copy()

filtered["browser"] = "Chrome"

# 5. Save filtered browsing history
os.makedirs("datas", exist_ok=True)
filtered.to_csv(output_file, index=False)

# 6. Display results
print("Total Chrome visits:", len(history))
print("Visits within RAM period:", len(filtered))
print("Filtered history date range:",
      filtered["timestamp"].min(), "to", filtered["timestamp"].max())
print("Saved to:", output_file)

RAM logging period: 2026-09-11 16:41:42 to 2026-09-13 20:00:13
Total Chrome visits: 7035
Visits within RAM period: 405
Filtered history date range: 2026-09-11 16:42:13 to 2026-09-13 19:52:04
Saved to: datas/browsing_history.csv


## category mapping

In [6]:
import pandas as pd
from pathlib import Path

data_dir = Path("datas")
data_dir.mkdir(exist_ok=True)

domain_category_map = pd.DataFrame({
    "domain": [
        "guvi.in",
        "youtube.com",
        "kick.com",
        "google.com",
        "v2.zenclass.in",
        "chatgpt.com",
        "drive.google.com",
        "myntra.com"
    ],
    "category": [
        "learning",
        "entertainment",
        "entertainment",
        "search",
        "learning",
        "researching",
        "productivity",
        "shopping"
    ]
})

domain_category_map.to_csv(
    data_dir / "domain_category_map.csv",
    index=False
)
